In [21]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

# --------------------------------------------
# 1) Load dataset
# --------------------------------------------
df = pd.read_csv("AAPL.csv")

print(df.head())
print(df.columns)

# --------------------------------------------
# 2) Clean + Sort
# --------------------------------------------
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# --------------------------------------------
# 3) Feature Engineering
# --------------------------------------------

# Date features
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Weekday"] = df["Date"].dt.weekday

# Lag features (Open prices)
df["Open_lag1"] = df["Open"].shift(1)
df["Open_lag2"] = df["Open"].shift(2)
df["Open_lag3"] = df["Open"].shift(3)
df["Open_lag5"] = df["Open"].shift(5)

# Target: next day's Open
df["Target"] = df["Open"].shift(-1)

# Remove NaN rows
df = df.dropna().reset_index(drop=True)

# --------------------------------------------
# 4) Select features
# --------------------------------------------
features = ["Open", "Year", "Month", "Day", "Weekday",
            "Open_lag1", "Open_lag2", "Open_lag3", "Open_lag5"]

X = df[features]
y = df["Target"]

# --------------------------------------------
# 5) Train-Test Split
# --------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# --------------------------------------------
# 6) Train CatBoost Model
# --------------------------------------------
model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=200
)

model.fit(X_train, y_train, eval_set=(X_test, y_test))

# --------------------------------------------
# 7) Predict and Evaluate
# --------------------------------------------
pred = model.predict(X_test)

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred)

print("\n📊 Model Performance:")
print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

# --------------------------------------------
# 8) Save model
# --------------------------------------------
model.save_model("catboost_stock_model.cbm")
joblib.dump(features, "features.pkl")

print("\n✅ Model saved as 'catboost_stock_model.cbm'")
print("✅ Feature list saved as 'features.pkl'")

# --------------------------------------------
# 9) Predict Next Day
# --------------------------------------------
last_row = df[features].iloc[-1:]
next_price = model.predict(last_row)[0]

print(f"\n🔮 Predicted next-day Open price: ${next_price:.2f}")


       Date       Open
0  1/2/2013  79.117142
1  1/3/2013  78.268570
2  1/4/2013  76.709999
3  1/7/2013  74.571426
4  1/8/2013  75.601425
Index(['Date', 'Open'], dtype='object')
0:	learn: 20.7228279	test: 56.1510060	best: 56.1510060 (0)	total: 2.39ms	remaining: 4.77s
200:	learn: 1.7168587	test: 32.8113949	best: 32.8113949 (200)	total: 245ms	remaining: 2.19s
400:	learn: 1.3423911	test: 32.7264594	best: 32.5886029 (267)	total: 470ms	remaining: 1.87s
600:	learn: 1.1044705	test: 32.8983350	best: 32.5886029 (267)	total: 702ms	remaining: 1.63s
800:	learn: 0.9216939	test: 32.9669206	best: 32.5886029 (267)	total: 974ms	remaining: 1.46s
1000:	learn: 0.7978284	test: 32.9551260	best: 32.5886029 (267)	total: 1.2s	remaining: 1.2s
1200:	learn: 0.6932970	test: 32.9803880	best: 32.5886029 (267)	total: 1.42s	remaining: 946ms
1400:	learn: 0.6073995	test: 33.0134625	best: 32.5886029 (267)	total: 1.65s	remaining: 706ms
1600:	learn: 0.5421417	test: 33.0219084	best: 32.5886029 (267)	total: 1.88s	remaining: 

In [23]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

df = pd.read_csv("AAPL.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print(df.head())
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Weekday"] = df["Date"].dt.weekday

df["Open_lag1"] = df["Open"].shift(1)
df["Open_lag2"] = df["Open"].shift(2)
df["Open_lag3"] = df["Open"].shift(3)
df["Open_lag5"] = df["Open"].shift(5)

df["Tomorrow"] = df["Open"].shift(-1)
df["UpDown"] = (df["Tomorrow"] > df["Open"]).astype(int)

df = df.dropna().reset_index(drop=True)

features = ["Open", "Year", "Month", "Day", "Weekday",
            "Open_lag1", "Open_lag2", "Open_lag3", "Open_lag5"]

X = df[features]
y = df["UpDown"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    loss_function="Logloss",
    eval_metric="Accuracy",
    verbose=200,
    random_seed=42
)

model.fit(X_train, y_train, eval_set=(X_test, y_test))

pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)

print("\n📊 Model Accuracy:", acc)
print("\nClassification Report:\n", classification_report(y_test, pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred))

model.save_model("catboost_stock_direction.cbm")
joblib.dump(features, "features_direction.pkl")

print("\n✅ Model saved as 'catboost_stock_direction.cbm'")
print("✅ Features saved as 'features_direction.pkl'")

last_row = df[features].iloc[-1:]
next_pred = model.predict(last_row)[0]

if next_pred == 1:
    print("\n🔮 Prediction: Price will go UP tomorrow")
else:
    print("\n🔮 Prediction: Price will go DOWN tomorrow")


        Date       Open
0 2013-01-02  79.117142
1 2013-01-03  78.268570
2 2013-01-04  76.709999
3 2013-01-07  74.571426
4 2013-01-08  75.601425
0:	learn: 0.5628743	test: 0.5139442	best: 0.5139442 (0)	total: 3.97ms	remaining: 3.97s
200:	learn: 0.8842315	test: 0.4860558	best: 0.5179283 (2)	total: 320ms	remaining: 1.27s
400:	learn: 0.9540918	test: 0.4860558	best: 0.5179283 (2)	total: 610ms	remaining: 911ms
600:	learn: 0.9790419	test: 0.4980080	best: 0.5179283 (2)	total: 926ms	remaining: 615ms
800:	learn: 0.9940120	test: 0.4940239	best: 0.5179283 (2)	total: 1.24s	remaining: 307ms
999:	learn: 1.0000000	test: 0.4900398	best: 0.5179283 (2)	total: 1.52s	remaining: 0us

bestTest = 0.5179282869
bestIteration = 2

Shrink model to first 3 iterations.

📊 Model Accuracy: 0.5179282868525896

Classification Report:
               precision    recall  f1-score   support

           0       0.44      0.48      0.46       107
           1       0.59      0.55      0.57       144

    accuracy            